# 📊 Adaptive Residual Steering (ARS) — Standalone Paper Evaluation Suite

This standalone notebook provides the definitive, publication-grade evaluation harness for **Adaptive Residual Steering (ARS)** across standard reasoning benchmarks.

### Core Capabilities:
1. **Checkpoint & Experiment Registry Loading**: Seamlessly load model weights from `EXPERIMENT_ID` or standalone `.pt` weight files.
2. **Multi-Dataset Benchmarks**: Standardized multi-step reasoning evaluation spanning:
   - **GSM8K** (`openai/gsm8k`): Grade school arithmetic.
   - **SVAMP** (`Chillee/SVAMP`): Variable-challenge word problems.
   - **ARC** (`allenai/ai2_arc`, ARC-Challenge): Multi-step scientific reasoning.
   - **MATH-500** (`HuggingFaceH4/MATH-500`): Competition high-school mathematics.
   - **GSM-Plus** (`qintong/GSM-Plus`): Adversarial & perturbed reasoning.
3. **Fair Multi-Shot Protocol**: Identical prompts evaluated across:
   - **Condition 1 (Frozen Baseline)**: Unsteered Phi-2 backbone.
   - **Condition 2 (SteerNet Only)**: Full SteerNet injection with Token Gate = 1.0.
   - **Condition 3 (Full ARS Pipeline)**: Dynamic attention-routed layer selection + Token Gating.
   - Evaluated across **0-shot, 2-shot, 4-shot, and 8-shot** with **Prompt Shielding**.
4. **Black-Orange Publication Plots**: Automatic generation of high-contrast, black-background `#000000` figures.
5. **LaTeX Summary Table Export**: Publication-ready LaTeX tables (`\begin{table*} ... \end{table*}`) formatted with `booktabs`.
6. **Experiment Registry Sync**: Automated telemetry recording and artifact indexing.

## 1. Environment Setup & GPU Telemetry

In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.36.0" "datasets>=2.16.0" "accelerate>=0.25.0" matplotlib

## 2. Workspace Setup & Repository Sync

In [ ]:
import os
import sys

REPO_DIR = '/kaggle/working/adaptive-residual-steering'
BRANCH = 'feature/workstream4-5-6-benchmarks-eval-diagrams'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Abdullah182155/adaptive-residual-steering.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    %cd /kaggle/working

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Workspace active at: {REPO_DIR}")

## 3. Weight Loading & Model Initialization

Specify an `EXPERIMENT_ID` from the registry or provide a direct path to a trained checkpoint.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.configs.config import RSCConfig
from src.models.wrapper import Phi2WithRSC
from src.experiments.registry import ExperimentRegistry, load_experiment

# User-specified checkpoint configuration
EXPERIMENT_ID = "latest"       # Set to specific ID like 'exp_20261008_1420_ars_phi2' or 'latest'
MANUAL_WEIGHTS_PATH = ""      # Optional fallback: e.g., '/kaggle/input/ars-checkpoints/final_ars_weights.pt'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

cfg = RSCConfig()
registry = ExperimentRegistry(base_dir=os.path.join(REPO_DIR, "experiments"))

experiment_run = None
target_weights_path = None

if EXPERIMENT_ID == "latest":
    recent = registry.list_experiments(limit=1)
    if recent:
        exp_id = recent[0]["experiment_id"]
        experiment_run = registry.get_experiment(exp_id)
        print(f"Resolved latest experiment: {exp_id}")
elif EXPERIMENT_ID:
    try:
        experiment_run = registry.get_experiment(EXPERIMENT_ID)
        print(f"Loaded experiment record: {EXPERIMENT_ID}")
    except Exception as e:
        print(f"Warning: Experiment '{EXPERIMENT_ID}' not in registry ({e}).")

if experiment_run and experiment_run.weights_path and os.path.exists(experiment_run.weights_path):
    target_weights_path = experiment_run.weights_path
elif MANUAL_WEIGHTS_PATH and os.path.exists(MANUAL_WEIGHTS_PATH):
    target_weights_path = MANUAL_WEIGHTS_PATH

print(f"Weights source: {target_weights_path}")

# Load Base Backbone Model
print(f"Loading backbone: {cfg.model_name}...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.float32 if device.type == "cpu" else torch.float16,
    trust_remote_code=True,
).to(device)

# Wrap with Adaptive Residual Steering
model = Phi2WithRSC(base_model, cfg, device=device).to(device)

# Restore trained parameters if available
if target_weights_path and os.path.exists(target_weights_path):
    loaded = torch.load(target_weights_path, map_location=device)
    if isinstance(loaded, dict) and "steer_state" in loaded:
        model.load_state_dict(loaded["steer_state"], strict=False)
    elif isinstance(loaded, dict):
        model.load_state_dict(loaded, strict=False)
    print(f"✅ Successfully loaded trained ARS weights from {target_weights_path}")
else:
    print("ℹ️ No existing checkpoint loaded. Operating with initialized architecture for baseline demonstration.")

## 4. Benchmark Scope Selection & Configuration

Select the evaluation scope (datasets, sample counts, few-shot numbers, prompt shielding).

In [ ]:
from src.evaluation.multi_dataset import EvalBenchmarkConfig

# Evaluation scope knobs
SELECTED_DATASETS = ["gsm8k", "svamp", "arc", "math500", "gsm_plus"]
SAMPLES_PER_DATASET = {
    "gsm8k": 100,
    "svamp": 100,
    "arc": 100,
    "math500": 50,
    "gsm_plus": 100,
}
SHOT_COUNTS = [0, 2, 4, 8]
ENABLE_PROMPT_SHIELDING = True
MAX_NEW_TOKENS = 512

eval_config = EvalBenchmarkConfig(
    datasets=SELECTED_DATASETS,
    samples_per_dataset=SAMPLES_PER_DATASET,
    shot_counts=SHOT_COUNTS,
    prompt_shielding=ENABLE_PROMPT_SHIELDING,
    max_new_tokens=MAX_NEW_TOKENS,
)

print("Configured Benchmark Scope:")
print(f"  Datasets: {eval_config.datasets}")
print(f"  Samples:  {eval_config.samples_per_dataset}")
print(f"  Shots:    {eval_config.shot_counts}")
print(f"  Prompt Shielding: {eval_config.prompt_shielding}")

## 5. Automated Fair Multi-Shot Evaluation

Executes identical test items across:
1. **Frozen Baseline** (`set_routing_override('none')`)
2. **SteerNet Only** (`set_routing_override('all')`, `gate=1.0`)
3. **Full ARS Pipeline** (Dynamic cooperative routing + Token Gates)

In [ ]:
from src.evaluation.multi_benchmark_runner import run_fair_multishot_benchmark

# If no active experiment run, create an evaluation record in the registry
if experiment_run is None:
    experiment_run = registry.create_experiment(
        tag="paper_eval_suite",
        cfg=cfg,
        initial_notes="Standalone paper evaluation suite run."
    )

# Run comprehensive benchmark suite
benchmark_output = run_fair_multishot_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=device,
    config=eval_config,
    conditions=("baseline", "steernet", "ars"),
    experiment_run=experiment_run,
    verbose=True,
)

benchmark_tree = benchmark_output["benchmark_results"]
print("✅ Benchmark evaluation complete!")

## 6. Automated Publication-Quality Visualizations (Black-Orange Palette)

Renders publication-grade charts strictly following the black-orange publication theme (`#000000` background, `#ff8000` vivid orange, `#999999` baseline gray).

In [ ]:
import matplotlib.pyplot as plt
from src.visualization.theme import set_ars_publication_theme
from src.visualization.plots import (
    plot_multishot_comparison,
    plot_synergy_heatmap,
    plot_layer_selection_and_k,
)

set_ars_publication_theme()
charts_dir = os.path.join(experiment_run.run_dir, "charts")
os.makedirs(charts_dir, exist_ok=True)

# 1. Multi-Shot Scaling Comparison Plots for each dataset
for ds_name in eval_config.datasets:
    norm_ds = ds_name.lower().replace("-", "_")
    if norm_ds in benchmark_tree:
        chart_path = os.path.join(charts_dir, f"multishot_scaling_{norm_ds}.png")
        single_ds_dict = {norm_ds: benchmark_tree[norm_ds]}
        fig = plot_multishot_comparison(single_ds_dict, save_path=chart_path)
        experiment_run.register_chart(f"multishot_scaling_{norm_ds}", chart_path)
        plt.show()

# 2. Consolidated Multi-Dataset Scaling Chart
full_chart_path = os.path.join(charts_dir, "multishot_scaling_all_benchmarks.png")
fig_all = plot_multishot_comparison(benchmark_tree, save_path=full_chart_path)
experiment_run.register_chart("multishot_scaling_all_benchmarks", full_chart_path)
plt.show()

# 3. Router Synergy Heatmap
if hasattr(model, "router") and hasattr(model.router, "get_normalized_synergy_matrix"):
    synergy_mat = model.router.get_normalized_synergy_matrix().detach().cpu().numpy()
    layer_labels = [f"L{idx}" for idx in cfg.target_layers]
    syn_path = os.path.join(charts_dir, "synergy_heatmap.png")
    fig_syn = plot_synergy_heatmap(synergy_mat, layer_labels=layer_labels, save_path=syn_path)
    experiment_run.register_chart("synergy_heatmap", syn_path)
    plt.show()

# 4. Layer Selection Rates & K-Distribution
if hasattr(model, "get_routing_diagnostics"):
    diag = model.get_routing_diagnostics()
    sel_rates = diag.get("selection_rates", {})
    k_dist = diag.get("k_distribution", {})
    if sel_rates and k_dist:
        diag_path = os.path.join(charts_dir, "routing_diagnostics.png")
        fig_diag = plot_layer_selection_and_k(sel_rates, k_dist, save_path=diag_path)
        experiment_run.register_chart("routing_diagnostics", diag_path)
        plt.show()

print(f"✅ All publication visual artifacts saved to {charts_dir}")

## 7. Publication-Ready LaTeX Table Generation

Generates clean LaTeX tables (`booktabs`) for immediate inclusion in academic conference submissions.

In [ ]:
from src.evaluation.multi_benchmark_runner import export_latex_summary_table

latex_table_code = export_latex_summary_table(
    benchmark_results=benchmark_tree,
    caption="Reasoning accuracy across multi-dataset benchmarks and few-shot demonstrations for Adaptive Residual Steering (ARS).",
    label="tab:ars_main_benchmarks",
)

tables_dir = os.path.join(experiment_run.run_dir, "tables")
os.makedirs(tables_dir, exist_ok=True)
tex_path = os.path.join(tables_dir, "benchmark_summary.tex")
with open(tex_path, "w", encoding="utf-8") as f:
    f.write(latex_table_code)

print(f"LaTeX Table written to: {tex_path}")
print("=" * 80)
print(latex_table_code)
print("=" * 80)

## 8. Experiment Registry Finalization & Summary

In [ ]:
experiment_run.finalize(
    status="completed",
    summary_notes=f"Evaluated on {len(eval_config.datasets)} benchmarks across {eval_config.shot_counts} shots."
)

print("🎉 Experiment Record Finalized!")
print(f"ID: {experiment_run.experiment_id}")
print(f"Record Path: {experiment_run.record_path}")
print(f"Charts Registered: {len(experiment_run.data.get('artifacts', {}).get('charts', []))}")